In [8]:
import os
import random
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

In [9]:
DATASET_ROOT = "dataset"
OUTPUT_SPLIT_DIR = "splits_final"

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
SEED = 42

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

os.makedirs(OUTPUT_SPLIT_DIR, exist_ok=True)

In [10]:
dataset_root = Path(DATASET_ROOT)

tornado_dir = dataset_root / "tornado"
nontornado_dir = dataset_root / "nontornado"
stresstest_dir = dataset_root / "stresstest"

print("Tornado exists:", tornado_dir.exists())
print("Non-tornado exists:", nontornado_dir.exists())
print("Stress test exists:", stresstest_dir.exists())

Tornado exists: True
Non-tornado exists: True
Stress test exists: True


In [11]:
def collect_tornado_records(folder_path):
    records = []
    for path in sorted(folder_path.rglob("*")):
        if path.is_file() and path.suffix.lower() in VALID_EXTENSIONS:
            records.append({
                "filepath": str(path.resolve()),
                "binary_label": 1,
                "class_name": "tornado",
                "subclass": "tornado"
            })
    return records

In [12]:
def collect_nontornado_records(folder_path):
    records = []
    for subclass_dir in sorted(folder_path.iterdir()):
        if not subclass_dir.is_dir():
            continue

        subclass_name = subclass_dir.name

        for path in sorted(subclass_dir.rglob("*")):
            if path.is_file() and path.suffix.lower() in VALID_EXTENSIONS:
                records.append({
                    "filepath": str(path.resolve()),
                    "binary_label": 0,
                    "class_name": "nontornado",
                    "subclass": subclass_name
                })
    return records

In [13]:
def collect_stresstest_records(folder_path):
    records = []

    has_subdirs = any(p.is_dir() for p in folder_path.iterdir())

    if has_subdirs:
        for subclass_dir in sorted(folder_path.iterdir()):
            if not subclass_dir.is_dir():
                continue

            subclass_name = subclass_dir.name

            for path in sorted(subclass_dir.rglob("*")):
                if path.is_file() and path.suffix.lower() in VALID_EXTENSIONS:
                    records.append({
                        "filepath": str(path.resolve()),
                        "binary_label": -1,   # not used in training
                        "class_name": "stresstest",
                        "subclass": subclass_name
                    })
    else:
        for path in sorted(folder_path.rglob("*")):
            if path.is_file() and path.suffix.lower() in VALID_EXTENSIONS:
                records.append({
                    "filepath": str(path.resolve()),
                    "binary_label": -1,
                    "class_name": "stresstest",
                    "subclass": "stresstest"
                })

    return records

In [14]:
tornado_records = collect_tornado_records(tornado_dir)
nontornado_records = collect_nontornado_records(nontornado_dir)
stresstest_records = collect_stresstest_records(stresstest_dir)

print("Tornado count:", len(tornado_records))
print("Non-tornado count:", len(nontornado_records))
print("Stress test count:", len(stresstest_records))

Tornado count: 913
Non-tornado count: 1007
Stress test count: 302


In [15]:
main_records = tornado_records + nontornado_records
main_df = pd.DataFrame(main_records)

print("Main dataset size:", len(main_df))
main_df.head()

Main dataset size: 1920


,filepath,binary_label,class_name,subclass
0,/homes/k858p487/TornadoDetectionResearch/datas...,1,tornado,tornado
1,/homes/k858p487/TornadoDetectionResearch/datas...,1,tornado,tornado
2,/homes/k858p487/TornadoDetectionResearch/datas...,1,tornado,tornado
3,/homes/k858p487/TornadoDetectionResearch/datas...,1,tornado,tornado
4,/homes/k858p487/TornadoDetectionResearch/datas...,1,tornado,tornado


In [16]:
#subclass stratified split

main_df["stratify_key"] = (
    main_df["binary_label"].astype(str) + "_" + main_df["subclass"].astype(str)
)

print(main_df["stratify_key"].value_counts().sort_index())

stratify_key
0_cloudy            164
0_fog               200
0_rain              160
0_rainbow            23
0_sandstorm         210
0_sunrise_sunset     30
0_sunshine           10
0_thunderstorm      210
1_tornado           913
Name: count, dtype: int64


In [17]:
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-8

# First split: train vs temp
train_df, temp_df = train_test_split(
    main_df,
    test_size=(1.0 - TRAIN_RATIO),
    stratify=main_df["stratify_key"],
    random_state=SEED
)

# Second split: val vs test
relative_test_ratio = TEST_RATIO / (VAL_RATIO + TEST_RATIO)

val_df, test_df = train_test_split(
    temp_df,
    test_size=relative_test_ratio,
    stratify=temp_df["stratify_key"],
    random_state=SEED
)

train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = val_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

print("Train size:", len(train_df))
print("Val size:", len(val_df))
print("Test size:", len(test_df))

Train size: 1343
Val size: 288
Test size: 289


In [18]:
train_path = os.path.join(OUTPUT_SPLIT_DIR, "train.csv")
val_path = os.path.join(OUTPUT_SPLIT_DIR, "val.csv")
test_path = os.path.join(OUTPUT_SPLIT_DIR, "test.csv")
stress_path = os.path.join(OUTPUT_SPLIT_DIR, "stress_test.csv")

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)
pd.DataFrame(stresstest_records).to_csv(stress_path, index=False)

print("Saved files:")
print(train_path)
print(val_path)
print(test_path)
print(stress_path)

Saved files:
splits_final/train.csv
splits_final/val.csv
splits_final/test.csv
splits_final/stress_test.csv


In [19]:
print("Train binary label counts:")
print(train_df["binary_label"].value_counts().sort_index())

print("\nVal binary label counts:")
print(val_df["binary_label"].value_counts().sort_index())

print("\nTest binary label counts:")
print(test_df["binary_label"].value_counts().sort_index())

Train binary label counts:
binary_label
0    705
1    638
Name: count, dtype: int64

Val binary label counts:
binary_label
0    151
1    137
Name: count, dtype: int64

Test binary label counts:
binary_label
0    151
1    138
Name: count, dtype: int64


In [20]:
print("Train subclass distribution:")
print(train_df.groupby(["binary_label", "subclass"]).size())

print("\nValidation subclass distribution:")
print(val_df.groupby(["binary_label", "subclass"]).size())

print("\nTest subclass distribution:")
print(test_df.groupby(["binary_label", "subclass"]).size())

Train subclass distribution:
binary_label  subclass      
0             cloudy            115
              fog               140
              rain              112
              rainbow            16
              sandstorm         147
              sunrise_sunset     21
              sunshine            7
              thunderstorm      147
1             tornado           638
dtype: int64

Validation subclass distribution:
binary_label  subclass      
0             cloudy             24
              fog                30
              rain               24
              rainbow             4
              sandstorm          31
              sunrise_sunset      5
              sunshine            2
              thunderstorm       31
1             tornado           137
dtype: int64

Test subclass distribution:
binary_label  subclass      
0             cloudy             25
              fog                30
              rain               24
              rainbow             3
  